# hoodini CLI Workflow Demo

This notebook runs all main parts of the `hoodini` CLI workflow, step by step, as implemented in the command line tool. Each cell corresponds to a logical section of the pipeline.

In [ ]:
# Patch asyncio for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()
print("Asyncio patched for Jupyter compatibility.")

## 1. Import Required Libraries
Import all necessary libraries and modules used in cli.py, including any custom modules.

In [ ]:
import os
import sys
import logging
import pandas as pd
import rich_click as click
import tomli
from pathlib import Path
from hoodini.config import load_default_config
from hoodini.utils.cli_helpers import MutuallyExclusiveOption
from hoodini.utils.core import validate_input_file, validate_domains
from hoodini.utils.logging_utils import console, logger, stage_header, stage_done, run_with_spinner

## 2. Parse Command-Line Arguments
Simulate or manually define the command-line arguments that would be passed to cli.py, and parse them as needed.

In [ ]:
# Initialize console and logger (these are already available as imports)
# Test logging
logger.info("Hoodini workflow notebook initialized")

## 3. Initialize Main Workflow Components
Set up any objects, configurations, or initial states required for the workflow.(THIS IS OPTIONAL)

In [ ]:
class ConfigObj(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__

# Initialize configuration and logging
config = load_default_config()

# Display loaded config for reference
console.print('Loaded configuration:', config)

config = ConfigObj(config)

## 4. Execute Core Workflow Logic
Run the main logic of the workflow, step by step, as implemented in cli.py.

## Initialize inputs

In [ ]:

# Run main workflow logic (simplified for notebook)
# 1. Parse input files
console.print(f'Parsing input file: {config.input_file}')


from hoodini.initialize import initialize_inputs

# Initialize the records DataFrame
records = initialize_inputs(
    input_path="cas9.txt",
    output=config.output,
)

In [ ]:
from hoodini.parse_ipg import run_ipg

records = run_ipg(
    records_df=records,  # or provide a DataFrame if you already have one
    cand_mode="best_id",)

console.print("IPG step complete. Records loaded.")

In [ ]:
from hoodini.parse_assemblies import run_assembly_parser
result = run_assembly_parser(
    records_df=records,
    output_dir=config.output,
    assembly_folder=config.assembly_folder,
    ncrna=config.ncrna,
    cctyper=config.cctyper,
    genomad=config.genomad,
    blast=config.blast,
    apikey=config.apikey,
    max_concurrent_downloads=config.max_concurrent_downloads,
    img=config.img,
    num_threads=config.num_threads,
    mod=config.mod,
    wn=config.wn,
    sorfs=config.sorfs,
    minwin=config.minwin,
    minwin_type=config.minwin_type,
)

In [ ]:
records = result["records"]
all_gff = result["all_gff"]
all_prots = result["all_prots"]
all_neigh = result["all_neigh"]
valid_uids = result["valid_uids"]

In [ ]:
all_prots

In [ ]:
from hoodini.cluster_proteins import cluster_proteins  # assuming it's saved here

cluster = cluster_proteins(
    all_prots,
    output_dir=config.output,
    clust_method=config.clust_method,
    sorfs=config.sorfs
)

In [ ]:
cluster

In [ ]:
from hoodini.taxonomy import parse_taxonomy_and_build_tree

tree_str, den_data = parse_taxonomy_and_build_tree(
    records=records,
    all_gff=all_gff,
    all_neigh=all_neigh,
    all_prots=all_prots,
    output_dir=config.output,
    tree_mode=config.tree_mode,
    tree_file=config.tree_file,
    num_threads=config.num_threads,
    aai_mode=config.aai_mode,
    ani_mode=config.ani_mode,
    aai_subset_mode=config.aai_subset_mode,
    nj_algorithm=config.nj_algorithm,
)

In [ ]:
from hoodini.protein_links import run_protein_links
pairwise_aa = run_protein_links(
    output_dir=config.output,
    all_prots=all_prots,
    threads=config.num_threads,
    evalue=1e-5
)

In [ ]:
from hoodini.pairwise_nt import run_pairwise_nt

pairwise_ani, nt_links = run_pairwise_nt(
    all_neigh=all_neigh,
    all_gff=all_gff,
    output_dir=config.output,
    nt_aln_mode="blastn",
    ani_mode="blastn",
    nt_links=True,
    threads=config.num_threads,
)

In [ ]:
from hoodini.extra_tools.domain import run_domain
# config.domains is already a validated list of database names
config.domains = ["pfam","phrog"]
domains_data = run_domain(all_prots, config.output, config.domains, config.num_threads)

In [ ]:
domains_data

In [ ]:
from hoodini.extra_tools.padloc import run_padloc
padloc_df = run_padloc(all_gff, all_prots, config.output, config.num_threads)
all_prots = all_prots.merge(padloc_df, on="id", how="left")
from hoodini.extra_tools.defensefinder import run_defensefinder
deffinder_df = run_defensefinder(all_gff, all_prots, config.output)
if not deffinder_df.empty:
    all_prots = all_prots.merge(deffinder_df, on="id", how="left")

In [ ]:

from hoodini.extra_tools.blast import run_blast
blast_data = run_blast(all_neigh, config.output, "query.fa", config.num_threads, valid_uids)
#convert to GFF-like format in which start = qstart, end=qend, ID=qseqid, feature type should be "region"
if not blast_data.empty:
    gff_df = pd.DataFrame({
        "seqid": blast_data["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": blast_data["start"],
        "end": blast_data["end"],
        "score": ".",
        "strand": "+",
        "phase": ".",
        "attributes": "ID=" + blast_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.cctyper import run_cctyper
cctyper_df, crispr_df = run_cctyper(all_gff, all_prots, all_neigh, config.output, config.num_threads, valid_uids)
#if cctyper_df is not empty:
if not cctyper_df.empty:
    all_prots = all_prots.merge(cctyper_df, on="id", how="left")
if not crispr_df.empty:
    gff_df = pd.DataFrame({
        "seqid": crispr_df["Contig"],
        "source": "hoodini",
        "type": "region",
        "start": crispr_df["start"],
        "end": crispr_df["end"],
        "score": ".",
        "strand": ".",
        "phase": ".",
        "attributes": "ID=" + crispr_df["nc_feature"] + ";"
    })
    #append to all_gff
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

from hoodini.extra_tools.genomad import run_genomad
genomad_df = run_genomad(all_neigh, config.output, config.num_threads, valid_uids)
if not genomad_df.empty:
    gff_df = pd.DataFrame({
        "seqid": genomad_df["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": genomad_df[["start", "end"]].min(axis=1),
        "end": genomad_df[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ".Z",
        "phase": ".",
        "attributes": "ID=" + genomad_df["mge_type"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.ncrna import run_ncrna
ncrna_data = run_ncrna(all_neigh, den_data, config.output, config.num_threads, valid_uids)
if not ncrna_data.empty:
    gff_df = pd.DataFrame({
        "seqid": ncrna_data["nucid"],
        "source": "hoodini",
        "type": "ncRNA",
        "start": ncrna_data[["start", "end"]].min(axis=1),
        "end": ncrna_data[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ncrna_data["strand_ncrna"],
        "phase": ".",
        "attributes": "ID=" + ncrna_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

In [ ]:
# Use the configured output directory directly
outdir = Path(config.output+ "/hoodini-viz")
outdir.mkdir(parents=True, exist_ok=True)

# GFF
all_gff.to_csv(outdir / "defaultGFF.gff", index=False, sep="\t", header=False)

# Baselines
all_neigh = all_neigh.rename(columns={
    "unique_id": "hood_id",
    "start_win": "start",
    "end_win": "end",
    "target_prot": "align_gene",
})
all_neigh[["hood_id", "seqid", "start", "end", "align_gene"]].to_csv(
    outdir / "defaultBaselines.txt", sep="\t", index=False
)

# Protein metadata
all_prots = all_prots.rename(columns={"protein_id": "gene_id", "fam_cluster": "cluster"})
all_prots["product"] = all_prots["product"].str.replace("\n", " ")
all_prots.to_csv(outdir / "defaultProteinMetadata.txt", sep="\t", index=False)

# Tree metadata
den_data = den_data.rename(columns={"unique_id": "leaf_id"})
den_data["leaf_id"] = den_data["leaf_id"].astype(str).str.extract(r"(\d+)").astype(int)
den_data.to_csv(outdir / "defaultTreeMetadata.txt", sep="\t", index=False)

# Newick tree
with open(outdir / "defaultNewick.txt", "w", encoding="utf-8") as f:
    f.write(tree_str)
    
# Domain metadata
if config.domains:
    with open(outdir / "defaultDomains.txt", "w", encoding="utf-8") as f:
        domains_data[["protein_id","domain_id","start","end","database"]].to_csv(f, sep="\t", index=False, header=False)
    
# Nucleotide link
# write nucleotide links; if nt_links not produced, write placeholder header-only file
if nt_links:
    nt_links[["query","query_start","query_end","ref","ref_start","ref_end","ani"]].to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["query","query_start","query_end","ref","ref_start","ref_end","ani"]).to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)

# Protein links
# pairwise_aa is only created when protein comparisons were run; if missing,
# write an empty placeholder file with the expected columns so downstream
# consumers always find the file.
if pairwise_aa:
    pairwise_aa[["qseqid","sseqid","pident"]].to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["qseqid","sseqid","pident"]).to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)

